[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umatter/EDFB/blob/main/notebooks/R/02_EAIF_Logistic_Regression.ipynb)

# EAIF - AI for Finance - Linear Probability Model and Logistic Regression

---

This notebook shows how to model a binary outcome in R. The business question is a classic one in retail banking: which clients should the bank call to sell a term deposit? We start with the linear probability model (OLS on a 0/1 outcome) to see where it breaks, move to logistic regression, and end with the decision the model is meant to support: whom to call, given the cost of a call and the revenue from a sale.

**Learning objectives.** After working through this notebook you can

1. explain why OLS on a 0/1 outcome (the linear probability model) is problematic, and when it is still fine;
2. fit a logistic regression and read its coefficients as odds ratios;
3. say which columns a bank knows at the moment it decides whom to call, and why the others must not be used;
4. judge a classifier with precision, recall and AUC against the majority-class baseline, and explain why accuracy alone misleads when only 11% of clients subscribe;
5. choose a classification threshold from the cost of a call and the revenue from a sale.

The data set `banking.csv` is loaded directly from GitHub; no download is needed.

Written for R 4.3 or newer (the Colab R runtime). Packages are installed as binaries from the Posit Package Manager, which keeps the setup cell fast.


In [ ]:
# Install and load the packages we need.
# Binary builds from the Posit Package Manager keep this cell fast on Colab.
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

required_packages <- c(
  "readr",     # reading CSV files
  "tidyr",     # reshaping data for plots
  "ggplot2",   # plots
  "gridExtra", # arranging several plots in one figure
  "corrplot",  # correlation heatmap
  "caret",     # train/test split, dummy variables, confusion matrix
  "pROC",      # ROC curves and AUC
  "dplyr"      # data manipulation (loaded last so that its select() is not masked)
)

new_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(new_packages) > 0) install.packages(new_packages, dependencies = TRUE, quiet = TRUE)

invisible(lapply(required_packages, library, character.only = TRUE))
cat("R version:", R.version.string, "\n")


In [ ]:
# To make this notebook's output stable across runs (we make the output reproducible)
set.seed(42)


## 0. Preparatory Steps

Before modelling we load the data, decide which columns the bank may use, look at the outcome and the features, encode the categorical variables, and split the data. This section covers:
- loading and exploring the banking data set
- decision-time feature selection (which columns are known when the bank decides whom to call?)
- the target variable and its base rate
- encoding categorical variables and standardising numerical ones
- correlated and collinear features
- the train/test split, and undersampling of the training set

### 0.1 Data Loading

We use a data set on direct-marketing phone campaigns of a Portuguese bank. Each row is one client who was called; the outcome `y` records whether the client subscribed to a term deposit (1) or not (0).


In [ ]:
# Import real data from GitHub
banking_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
print("Fetching banking.csv from GitHub...")


In [ ]:
dataset <- read_csv(banking_url, show_col_types = FALSE)


### 0.2 Initial Data Exploration

Let's examine the structure and basic properties of our dataset.


In [ ]:
# Check dataset dimensions
dim(dataset)


In [ ]:
# Examine data types of all variables
str(dataset)


### 0.3 Variable Classification

We separate our features into numerical and categorical variables for appropriate preprocessing.


In [ ]:
# Define set of numerical and categorical variables
num_var <- names(select_if(dataset %>% select(-y), is.numeric))
cat_var <- names(select_if(dataset %>% select(-y), function(x) is.character(x) | is.factor(x)))


In [ ]:
print("Numerical variables:")
print(num_var)


In [ ]:
print("Categorical variables:")
print(cat_var)


### 0.4 Data Quality Assessment


In [ ]:
# Check for missing values
sapply(dataset, function(x) any(is.na(x)))


In [ ]:
# Get basic statistics for numerical variables
summary(dataset[num_var])


### 0.5 Which columns does the bank know at decision time?

The model will be used to decide whom to call. So, following the rule from unit 01, the only features we may use are those the bank knows at the moment that decision is made. Ask this question of every column before you use it.

`duration` is the length of the phone call in seconds. It is an excellent predictor of `y`: long calls end in a sale, short calls do not. But the bank only learns the duration after the call has ended, and at that point it also knows the outcome. A model with `duration` would look very accurate on historical data and be useless for the decision. This is called data leakage, and it is one of the most common mistakes in applied prediction work.

The columns that describe earlier campaigns are known before the call and may be used: `poutcome` (the outcome of the previous campaign) and `previous` (the number of earlier contacts). We keep `poutcome`.

We drop a few more columns for simplicity, not because they leak:
- `pdays` (days since the last contact) uses the code 999 for "never contacted before". That is a code, not missing data, and the column could be recoded as a previously-contacted indicator. We drop it to keep the example short.
- `campaign` counts the contacts in the current campaign including the current call, so it is only partly known at decision time.
- `age` and `previous` are dropped to keep the feature set small.


In [ ]:
# duration: known only after the call has ended (leakage)
# pdays: 999 is a code for "never contacted"; dropped for simplicity
# campaign: includes the current call; age, previous: dropped for simplicity
dataset <- dataset %>% select(-duration, -pdays, -age, -campaign, -previous)
num_var <- names(select_if(dataset %>% select(-y), is.numeric))
print("Remaining numerical variables:")
print(num_var)


### 0.6 The target variable and its base rate

Only a minority of clients subscribe. The share of ones in `y` is the base rate. It matters twice: it is the accuracy a model reaches by always predicting "no" (the majority-class baseline), and it is the starting point for every probability the model will produce.


In [ ]:
# Distribution of the target variable (y: will the client subscribe?)
base_rate <- mean(dataset$y)
cat(sprintf("Subscribers: %d of %d clients (%.1f%%)\n", sum(dataset$y), nrow(dataset), 100 * base_rate))
cat(sprintf("Majority-class baseline: always predicting 'no' gives accuracy %.3f\n", 1 - base_rate))

ggplot(dataset, aes(x = factor(y))) +
  geom_bar(fill = "steelblue", color = "black") +
  labs(title = "Distribution of the target variable", x = "Subscription (y)", y = "Count") +
  theme_minimal()


### 0.7 Exploratory Data Analysis: features by outcome

One figure is enough to see which features separate subscribers from non-subscribers. The top panels show the distribution of each numerical feature for the two groups. The panels below show the share of subscribers within each level of a categorical feature, with the overall base rate as a dashed line. A level far from the dashed line is informative.


In [ ]:
# Numerical features: distribution by outcome
plot_num <- dataset %>%
  select(all_of(num_var), y) %>%
  pivot_longer(-y, names_to = "variable", values_to = "value") %>%
  ggplot(aes(x = value, fill = factor(y))) +
  geom_density(alpha = 0.5) +
  facet_wrap(~variable, scales = "free", ncol = 5) +
  scale_fill_discrete(name = "", labels = c("Not subscribed (0)", "Subscribed (1)")) +
  labs(title = "Numerical features by outcome", x = "", y = "density") +
  theme_minimal() +
  theme(legend.position = "bottom")

# Categorical features: share of subscribers per level
plot_cat <- dataset %>%
  select(all_of(cat_var), y) %>%
  pivot_longer(-y, names_to = "variable", values_to = "level") %>%
  group_by(variable, level) %>%
  summarise(share_subscribed = mean(y), .groups = "drop") %>%
  ggplot(aes(x = level, y = share_subscribed)) +
  geom_col(fill = "steelblue") +
  geom_hline(yintercept = base_rate, linetype = "dashed", color = "red") +
  facet_wrap(~variable, scales = "free_x", ncol = 3) +
  labs(title = "Share of subscribers per level (dashed line: overall base rate)", x = "", y = "share subscribed") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

options(repr.plot.width = 14, repr.plot.height = 12)
grid.arrange(plot_num, plot_cat, ncol = 1, heights = c(1, 2))


### 0.8 Data Preprocessing

Regression models need numbers. We encode each categorical variable as a set of 0/1 dummy variables, dropping one level per variable as the reference category (`fullRank = TRUE`; otherwise the dummies of one variable would add up to the constant). The numerical features will be standardised to mean 0 and standard deviation 1, so that their coefficients read as "per standard deviation". We do that in section 0.10, after the split, with the training rows' means and standard deviations only: the test rows must not influence any step of the fit, not even the scaling.


In [ ]:
# Dummy variables for categorical features (one reference level dropped per variable)
# Note: caret names the dummies with an underscore (loan_unknown), while model.matrix in unit 01 named them
# without one (loanunknown); it is the same variable under two spellings.
X_raw <- dataset %>% select(-y) %>% mutate(across(all_of(cat_var), as.factor))
dummies <- dummyVars(~ ., data = X_raw, fullRank = TRUE, sep = "_")
dataset_dummy <- as.data.frame(predict(dummies, newdata = X_raw))
dataset_dummy$y <- dataset$y


In [ ]:
head(dataset_dummy)


### 0.9 Correlated and collinear features

Two features that carry the same information make a regression unstable: the model cannot tell which of the two deserves the coefficient. We list the pairs with the highest absolute correlation, draw the heatmap, and keep one feature of each highly correlated group.

Two things stand out in the list below.

- `housing_unknown` and `loan_unknown` have correlation 1. They are the same column: the 990 clients with unknown housing-loan status are exactly the clients with unknown personal-loan status. This is perfect collinearity, the extreme case of the multicollinearity we are looking for. With both columns in the model there is no unique solution: R's `lm()` reports one of them as `NA`, and scikit-learn silently splits the coefficient between the two. We keep `housing_unknown` and drop `loan_unknown`.
- The macro indicators `emp_var_rate`, `euribor3m`, `nr_employed` and `cons_price_idx` move together (pairwise correlations between 0.5 and 0.97), because they all measure the state of the economy at the time of the call. We drop those four and keep `cons_conf_idx` (consumer confidence), which is only weakly correlated with the rest (below 0.3).


In [ ]:
# Correlation matrix, lower triangle only, listed as pairs sorted by absolute correlation
corrmat <- cor(dataset_dummy %>% select(-y))
corrmat_lower <- corrmat
corrmat_lower[upper.tri(corrmat_lower, diag = TRUE)] <- NA

corrmat_df <- as.data.frame(as.table(corrmat_lower))
corrmat_df <- corrmat_df[!is.na(corrmat_df$Freq), ]
names(corrmat_df) <- c("feature_1", "feature_2", "correlation")
corrmat_df$abs_corr <- abs(corrmat_df$correlation)
corrmat_df <- corrmat_df[order(-corrmat_df$abs_corr), ]
head(corrmat_df[, c("feature_1", "feature_2", "correlation")], 10)


In [ ]:
# Correlation heatmap
options(repr.plot.width = 12, repr.plot.height = 12)
corrplot(corrmat, method = "color", type = "upper", order = "hclust",
         tl.cex = 0.8, tl.col = "black", tl.srt = 45)


In [ ]:
# Drop one feature of each highly correlated group (see the list above)
col_to_drop <- c("emp_var_rate", "cons_price_idx", "euribor3m", "nr_employed", "loan_unknown")
dataset <- dataset_dummy %>% select(-all_of(col_to_drop))
num_var <- intersect(num_var, names(dataset))

# Ready to train and test our models!
X <- dataset %>% select(-y)
y <- dataset$y
cat("Feature matrix dimensions:", nrow(X), "x", ncol(X), "\n")
cat("Target vector length:", length(y), "\n")


### 0.10 Train/test split, then undersampling of the training set

We first split the data into a training set (80%) and a test set (20%), stratified on `y` so that both have the natural share of subscribers (about 11.3%). The test set is not touched again until we evaluate: it stands in for the clients the bank will call next. The numerical features are standardised right after the split, with the training rows' means and standard deviations applied to both sets, so that nothing about the test rows enters the fit.

Only 11% of the training rows are subscribers. As a teaching device we undersample the training set: we keep every subscriber and a random sample of non-subscribers twice as large, so the model is trained on a 1:2 mix. Three facts about this step matter.

- For logistic regression, undersampling the majority class shifts the intercept by `log(sampling ratio)`, where the sampling ratio is the share of non-subscribers we kept. The other coefficients are unchanged in expectation.
- Because only the intercept moves, the ranking of clients by predicted probability is unchanged, and so is the AUC.
- The probabilities from the undersampled fit are not deployment probabilities. They are too high by construction, because the model was shown a world in which a third of clients subscribe. We recover deployment probabilities in section 2.4 by shifting the intercept back.

Everything is evaluated on the untouched test set with its natural 11.3% base rate.


In [ ]:
# Split train and test set, stratified on y (createDataPartition stratifies only on a factor)
set.seed(0)
train_indices <- createDataPartition(factor(dataset$y), p = 0.8, list = FALSE)
X_train <- X[train_indices, ]
X_test <- X[-train_indices, ]
y_train <- y[train_indices]
y_test <- y[-train_indices]

# Standardise numerical features with the TRAINING rows' means and standard deviations (mean 0, SD 1 on the training set)
train_means <- colMeans(X_train[num_var])
train_sds <- sapply(X_train[num_var], sd)
X_train[num_var] <- as.data.frame(scale(X_train[num_var], center = train_means, scale = train_sds))   # as.data.frame() keeps plain numeric columns
X_test[num_var] <- as.data.frame(scale(X_test[num_var], center = train_means, scale = train_sds))

cat(sprintf("Training set: %d rows, share subscribed %.3f\n", nrow(X_train), mean(y_train)))
cat(sprintf("Test set:     %d rows, share subscribed %.3f\n", nrow(X_test), mean(y_test)))
cat("Scaler fit on the training rows: means", round(train_means, 3), "; SDs", round(train_sds, 3), "\n")


In [ ]:
# Undersample the TRAINING set only: keep all subscribers and twice as many non-subscribers
train_data <- X_train
train_data$y <- y_train
train_1 <- train_data %>% filter(y == 1)
train_0 <- train_data %>% filter(y == 0)
set.seed(0)
train_0_small <- train_0 %>% slice_sample(n = 2 * nrow(train_1))
train_bal <- bind_rows(train_1, train_0_small) %>% slice_sample(prop = 1)

X_train_bal <- train_bal %>% select(-y)
y_train_bal <- train_bal$y
sampling_ratio <- nrow(train_0_small) / nrow(train_0)   # share of non-subscribers kept

cat(sprintf("Subscribers in training set: %d, non-subscribers: %d, kept: %d\n", nrow(train_1), nrow(train_0), nrow(train_0_small)))
cat(sprintf("Undersampled training set: %d rows, share subscribed %.3f\n", nrow(X_train_bal), mean(y_train_bal)))
cat(sprintf("Sampling ratio: %.3f, so the intercept shifts by log(%.3f) = %.3f\n", sampling_ratio, sampling_ratio, log(sampling_ratio)))


## 1. Linear Probability Model (LPM)

Before diving into logistic regression, let's start with the simpler Linear Probability Model. The LPM treats the binary dependent variable as if it were continuous and applies ordinary least squares (OLS) regression.

**Model specification:** P(y=1|X) = β₀ + β₁X₁ + β₂X₂ + ... + βₖXₖ + ε

While conceptually simple, the LPM has several important limitations that we'll explore.


In [ ]:
# Fit Linear Probability Model using OLS on the undersampled training set
# In R, we use lm() for linear regression
lpm_model <- lm(y ~ ., data = train_bal)

# Get predictions
y_train_pred_lpm <- predict(lpm_model, X_train_bal)
y_test_pred_lpm <- predict(lpm_model, X_test)

# R² is hard to interpret for a 0/1 outcome; we use classification metrics from here on
cat(sprintf("LPM Training R²: %.4f\n", summary(lpm_model)$r.squared))


In [ ]:
# Examine LPM predictions and identify problems
cat("Linear Probability Model - Prediction Statistics:\n")
cat(sprintf("Training set predictions - Min: %.4f, Max: %.4f\n",
            min(y_train_pred_lpm), max(y_train_pred_lpm)))
cat(sprintf("Test set predictions - Min: %.4f, Max: %.4f\n",
            min(y_test_pred_lpm), max(y_test_pred_lpm)))

cat("\nPredictions outside [0,1] range:\n")
train_outside <- sum(y_train_pred_lpm < 0 | y_train_pred_lpm > 1)
test_outside <- sum(y_test_pred_lpm < 0 | y_test_pred_lpm > 1)

cat(sprintf("Training: %d out of %d (%.1f%%)\n",
            train_outside, length(y_train_pred_lpm),
            100 * train_outside / length(y_train_pred_lpm)))
cat(sprintf("Test: %d out of %d (%.1f%%)\n",
            test_outside, length(y_test_pred_lpm),
            100 * test_outside / length(y_test_pred_lpm)))


In [ ]:
# Visualize LPM predictions vs actual values
options(repr.plot.width = 14, repr.plot.height = 6)

# Plot 1: Histogram of predicted probabilities
p1 <- ggplot(data.frame(pred = y_test_pred_lpm), aes(x = pred)) +
  geom_histogram(bins = 30, fill = "lightblue", color = "black", alpha = 0.7) +
  geom_vline(xintercept = 0, color = "red", linetype = "dashed", linewidth = 1) +
  geom_vline(xintercept = 1, color = "red", linetype = "dashed", linewidth = 1) +
  labs(title = "Distribution of LPM Predicted Probabilities",
       x = "Predicted Probability", y = "Frequency") +
  theme_minimal()

# Plot 2: Scatter plot of predictions vs actual
p2 <- ggplot(data.frame(pred = y_test_pred_lpm, actual = y_test), aes(x = pred, y = actual)) +
  geom_point(alpha = 0.6) +
  geom_abline(intercept = 0, slope = 1, color = "red", linetype = "dashed") +
  labs(title = "LPM: Predicted vs Actual Values",
       x = "Predicted Probability (LPM)", y = "Actual Value") +
  theme_minimal()

grid.arrange(p1, p2, ncol = 2)


In [ ]:
# Convert LPM predictions to binary classifications (using 0.5 threshold)
y_test_pred_lpm_binary <- ifelse(y_test_pred_lpm >= 0.5, 1, 0)

# Accuracy, always next to the majority-class baseline
lpm_accuracy <- mean(y_test == y_test_pred_lpm_binary)
baseline_accuracy <- max(mean(y_test), 1 - mean(y_test))
cat(sprintf("LPM Classification Accuracy:                     %.4f\n", lpm_accuracy))
cat(sprintf("Majority-class baseline (always predict 'no'):   %.4f\n", baseline_accuracy))


## Problems with the Linear Probability Model

The LPM has real limitations when the outcome is 0/1:

### 1. Predicted probabilities outside [0, 1]
- As the diagnostic above shows, the LPM can predict values below 0 or above 1 (in this run only a few test predictions exceed 1, because our feature set has few strong predictors; with more of them the share grows)
- This violates the basic definition of a probability

### 2. Heteroskedasticity
- The error variance is not constant: Var(ε|X) = P(X)[1 − P(X)]
- The usual OLS standard errors are therefore wrong. The fix is routine: heteroskedasticity-robust standard errors (`sandwich::vcovHC()` in R)

### 3. Linear relationship assumption
- The LPM assumes a linear effect of X on P(y=1|X), so the marginal effect is the same at every value of X
- Near 0 or 1 this cannot hold: a probability of 0.98 cannot rise by another 0.05

### 4. Distributional assumptions
- The errors of a 0/1 outcome are Bernoulli, not normal. This matters only for exact small-sample inference, not for unbiasedness or consistency, so it is the least important point on this list

### 5. Efficiency
- Because of the heteroskedasticity, OLS is not the most efficient estimator; maximum likelihood (logistic regression) is

**Solution:** logistic regression keeps the probability inside [0, 1], models an S-shaped relationship, and is estimated by maximum likelihood.

### When the LPM is fine
- When you want coefficients that read directly as changes in probability (a coefficient of 0.05 means 5 percentage points) and the predicted probabilities stay well inside [0, 1]. For this reason the LPM with robust standard errors is the standard choice in applied economics for causal designs with fixed effects or instrumental variables, where the logit is awkward.
- When the goal is classification at a threshold: the LPM and the logit usually agree on almost every case (Exercise 1 lets you check this).


## 2. Logistic Regression

### 2.1 The model, log-odds and odds ratios

Logistic regression keeps the linear index of the LPM but passes it through the logistic (sigmoid) function, so that the output is always between 0 and 1:

P(y=1|X) = 1 / (1 + e^(−z)),  with  z = β₀ + β₁X₁ + ... + βₖXₖ

Three ways of reading the same equation:

- **Log-odds.** Solving for z gives z = log( P / (1 − P) ). The linear index is the log of the odds of subscribing. A coefficient βⱼ is the change in log-odds for a one-unit increase in Xⱼ. Log-odds are hard to feel, so nobody reports them directly.
- **Odds ratios.** Taking exponentials, a one-unit increase in Xⱼ multiplies the odds by exp(βⱼ). This is the odds ratio. An odds ratio of 2 doubles the odds, an odds ratio of 0.5 halves them; the two are equally large effects in opposite directions, which is why we sort by |β| (the absolute log odds ratio) and not by the odds ratio itself. For a 0/1 dummy the "one unit" is having the characteristic; for our standardised numerical features it is one standard deviation.
- **Probabilities.** The effect of Xⱼ on the probability itself is not constant: it depends on where on the S-curve the client sits. Near P = 0.5 the curve is steep and a change in z moves the probability a lot; near 0 or 1 the curve is flat and the same change barely matters. That is what the next plot shows.


In [ ]:
# The logistic (sigmoid) function: log-odds z on the x-axis, probability on the y-axis
options(repr.plot.width = 6, repr.plot.height = 3.5)
z <- seq(-6, 6, length.out = 200)
ggplot(data.frame(z = z, p = plogis(z)), aes(x = z, y = p)) +
  geom_line(color = "steelblue", linewidth = 1) +
  geom_hline(yintercept = 0.5, linetype = "dashed", color = "grey50") +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  labs(title = "The logistic (sigmoid) function", x = "log-odds z = b0 + b1*X1 + ...", y = "P(y = 1 | X)") +
  theme_minimal()


In [ ]:
# Fit Logistic Regression using glm() with family = binomial (maximum likelihood, no penalty)
logit_model <- glm(y ~ ., data = train_bal, family = binomial)

# The summary is the inference table: coefficients (log-odds), standard errors, z values and p-values
summary(logit_model)


In [ ]:
# Odds ratios, sorted by the absolute log-odds coefficient (an odds ratio of 0.5 is as large an effect as 2)
odds_table <- data.frame(
  feature = names(coef(logit_model)),
  log_odds_coef = unname(coef(logit_model)),
  odds_ratio = unname(exp(coef(logit_model)))
) %>%
  filter(feature != "(Intercept)") %>%
  arrange(desc(abs(log_odds_coef)))
cat(sprintf("Intercept (log-odds): %.3f\n", coef(logit_model)["(Intercept)"]))
print(odds_table, digits = 3, row.names = FALSE)


### 2.2 Reading the coefficient table

Two worked examples from the table above (numbers from this run; yours may differ slightly).

- `poutcome_success` is a dummy: 1 if the client subscribed in the previous campaign. Its coefficient is about 2.4 on the log-odds scale, so the odds ratio is exp(2.4) ≈ 11: a client who said yes last time has roughly 11 times the odds of saying yes again, other things equal. It is by far the strongest feature.
- `cons_conf_idx` is standardised, so its coefficient (about 0.14) is per standard deviation of consumer confidence (one SD is about 4.6 index points). The odds ratio exp(0.14) ≈ 1.15: one standard deviation more consumer confidence raises the odds of a sale by about 15%.

Two cautions:

- These are odds ratios, not probability changes. What an odds ratio of 11 means in percentage points depends on the client's starting probability: for a client at 5% it moves the probability to about 36%, for a client at 50% to about 92%. (Check: the odds go from 0.053 to 0.57, and from 1 to 11.)
- The intercept is shifted by the undersampling (section 0.10); the slope coefficients, and hence the odds ratios, are not.

The inference table (`summary()` in R) adds standard errors and p-values. Small levels such as `education_illiterate` (a handful of clients) have wide confidence intervals; treat their coefficients as noise.


### 2.3 Predictions, accuracy and the baseline

We now predict on the untouched test set. The default rule classifies a client as a subscriber if the predicted probability is at least 0.5.

**Which class is "positive"?** Throughout, y = 1 (the client subscribes) is the positive class. Precision, recall and the confusion matrix below refer to this class: precision is the share of clients we call (predicted 1) who actually subscribe, recall is the share of actual subscribers we reach.

**Why accuracy is the wrong headline number here.** About 88.7% of the test clients do not subscribe, so a rule that never calls anyone is 88.7% accurate. Every accuracy must be read next to that baseline. Precision, recall and the ROC curve tell us what the model adds; the confusion matrix is where they come from. The area under the ROC curve (AUC) is the probability that a randomly chosen subscriber gets a higher predicted probability than a randomly chosen non-subscriber.


In [ ]:
# Predicted probabilities and 0/1 predictions on the test set
y_test_predicted_prob_logit <- predict(logit_model, X_test, type = "response")
y_test_predicted_logit <- ifelse(y_test_predicted_prob_logit >= 0.5, 1, 0)

# Compare LPM vs Logistic Regression predictions
comparison_df <- data.frame(
  True = y_test,
  LPM_prob = y_test_pred_lpm,
  LPM_pred = y_test_pred_lpm_binary,
  Logit_prob = y_test_predicted_prob_logit,
  Logit_pred = y_test_predicted_logit
)
print(head(comparison_df, 20))

cat("\nModel Comparison (test set, threshold 0.5):\n")
cat(sprintf("LPM Accuracy:                                    %.4f\n", mean(y_test == y_test_pred_lpm_binary)))
cat(sprintf("Logistic Regression Accuracy:                    %.4f\n", mean(y_test == y_test_predicted_logit)))
cat(sprintf("Majority-class baseline (always predict 'no'):   %.4f\n", baseline_accuracy))


In [ ]:
# Confusion matrix for Logistic Regression: rows = true class (0, 1), columns = predicted class (0, 1), as in the Python version
table(Actual = y_test, Predicted = y_test_predicted_logit)


In [ ]:
# Plot the confusion matrix
plot_confusion_matrix <- function(y_true, y_pred, title = "Confusion Matrix") {
  # Create confusion matrix
  cm <- table(Actual = y_true, Predicted = y_pred)

  # Convert to data frame for ggplot
  cm_df <- as.data.frame(cm)

  # Create plot (true class on the y-axis, predicted class on the x-axis, as in the Python version)
  p <- ggplot(cm_df, aes(x = Predicted, y = Actual, fill = Freq)) +
    geom_tile(color = "white") +
    geom_text(aes(label = Freq), size = 12, color = "white") +
    scale_fill_gradient(low = "lightblue", high = "darkblue") +
    labs(title = title, x = "Predicted label", y = "True label") +
    theme_minimal() +
    theme(legend.position = "none")

  return(p)
}

options(repr.plot.width = 6, repr.plot.height = 5)
# Plot confusion matrix for Logistic Regression
plot_confusion_matrix(y_test, y_test_predicted_logit, "Logistic Regression Confusion Matrix")


In [ ]:
# Precision (Pos Pred Value), recall (Sensitivity) and more on the test set.
# positive = "1" tells caret that the subscribers are the positive class; without it caret
# would take the first factor level ("0") as positive and all rates would refer to non-subscribers.
cm <- confusionMatrix(factor(y_test_predicted_logit, levels = c(0, 1)),
                      factor(y_test, levels = c(0, 1)), positive = "1")
print(cm)


In [ ]:
# ROC curve and AUC. The AUC is always computed from predicted probabilities, never from 0/1 predictions.
roc_obj <- roc(y_test, y_test_predicted_prob_logit, levels = c(0, 1), direction = "<", quiet = TRUE)
auc_value <- auc(roc_obj)

# legacy.axes = TRUE puts 1 - specificity (the false positive rate) on the x-axis, as in the Python plot,
# so the dashed chance line runs from bottom-left to top-right
options(repr.plot.width = 6, repr.plot.height = 6)
plot(roc_obj, legacy.axes = TRUE, main = "Receiver Operating Characteristic",
     col = "blue", lwd = 2, xlab = "False Positive Rate", ylab = "True Positive Rate")
abline(a = 0, b = 1, col = "red", lty = 2)
legend("bottomright",
       legend = paste("Logistic Regression (AUC =", round(auc_value, 2), ")"),
       col = "blue", lwd = 2)


### 2.4 From the undersampled fit to deployment probabilities

Section 0.10 claimed that undersampling only shifts the intercept, by log(sampling ratio). We can check this directly: adding log(sampling ratio) to the model's log-odds and applying the sigmoid gives probabilities that are calibrated to the natural base rate. The AUC does not change, because adding the same constant to every client's log-odds does not change their ranking.

`p_test` below holds these deployment probabilities. The exercises use them, because the cost/revenue decision needs a probability that means what it says.

Note the consequence for the 0.5 threshold above: a threshold of 0.5 on the undersampled fit corresponds to about 0.20 on the deployment scale (in this run, sigmoid(log(0.254)) ≈ 0.20). "Predict 1 if p ≥ 0.5" was never a neutral choice.


In [ ]:
# Log-odds from the undersampled fit, shifted back by log(sampling_ratio), then the sigmoid
logodds_test <- predict(logit_model, X_test, type = "link") + log(sampling_ratio)
p_test <- plogis(logodds_test)   # deployment probabilities

cat(sprintf("Sampling ratio %.3f, intercept shift log(ratio) = %.3f\n", sampling_ratio, log(sampling_ratio)))
cat(sprintf("Mean predicted probability, undersampled fit: %.3f\n", mean(y_test_predicted_prob_logit)))
cat(sprintf("Mean predicted probability, corrected:        %.3f\n", mean(p_test)))
cat(sprintf("Share of subscribers in the test set:         %.3f\n", mean(y_test)))
cat(sprintf("AUC, undersampled fit: %.4f\n", auc(roc(y_test, y_test_predicted_prob_logit, levels = c(0, 1), direction = "<", quiet = TRUE))))
cat(sprintf("AUC, corrected:        %.4f\n", auc(roc(y_test, p_test, levels = c(0, 1), direction = "<", quiet = TRUE))))
cat(sprintf("Threshold 0.5 on the undersampled fit = %.3f on the deployment scale\n", plogis(log(sampling_ratio))))


## 3. Exercises

The exercises follow the campaign decision: whom should the bank call? Each exercise states its deliverable. Solutions are in `notebooks/R/solutions/02_EAIF_Logistic_Regression_Solutions.ipynb`.

### Exercise 1: LPM versus logit predictions

**Task:** Compare the two models' predicted probabilities on the test set.

**Instructions:**
1. Scatter `y_test_pred_lpm` (x-axis) against `y_test_predicted_prob_logit` (y-axis) and add the 45-degree line.
2. Compute the share of test clients on which the two models make the same 0/1 classification at threshold 0.5.
3. Find the client with the largest absolute difference between the two probabilities and look at its two predictions.

**Deliverable:** the agreement share, the largest difference, and one sentence on where along the probability range the models disagree.


In [ ]:
# Exercise 1: Your code here
# Hint: ggplot() with geom_point() for the two probability vectors, geom_abline() for the 45-degree line

# Your solution:



### Exercise 2: Threshold sweep

**Task:** The bank can move the threshold. See what each choice does to the campaign.

**Business context:** A low threshold means calling many clients (high recall, low precision); a high threshold means calling few (higher precision, low recall). Neither is right by itself.

**Instructions:**
1. For each threshold in `thresholds` (0.01 to 0.50), classify a test client as "call" if the deployment probability `p_test` is at least the threshold.
2. Compute the number of calls, the precision and the recall at each threshold.
3. Plot precision, recall and the number of calls against the threshold.

**Deliverable:** the plot, a three-row table (thresholds 0.05, 0.10 and 0.20) with calls, precision and recall, and one sentence on the trade-off you see.


In [ ]:
# Exercise 2: Your code here
# Hint: for each threshold t, call <- p_test >= t; precision and recall follow from the true positives

thresholds <- round(seq(0.01, 0.50, by = 0.01), 2)   # round() so that 0.10 is exactly 0.10

# Your solution:



### Exercise 3: Expected profit and the optimal threshold

**Task:** Choose the threshold from the economics of the campaign.

**Scenario:**
- Cost of one call: €5
- Revenue from one subscription: €100
- The bank has 10,000 clients like those in the test set. The test set has the natural base rate (about 11.3%), so it is representative of them.

**Instructions:**
1. For each threshold in `thresholds` (0.01 to 0.50, defined again in the cell below), compute the profit on the test set from calling every client with `p_test` at or above the threshold: revenue × true positives − cost × calls. Scale it to 10,000 clients (multiply by 10,000 / number of test clients).
2. Compare with the profit from calling everyone (no model).
3. Derive the profit-maximising rule by hand: a call is worth making when its expected revenue exceeds its cost, that is when p × €100 > €5. Compare the resulting threshold with the empirical optimum in your table.
4. Repeat steps 1 and 2 with a call cost of €20. Where is the break-even now, and how much does the model add compared with calling everyone?

**Deliverable:** the best threshold and profit per 10,000 clients for both costs, the "call everyone" profit for both, the rule `p > cost / revenue` in one sentence, and one sentence on why the rule needs the deployment probabilities `p_test` rather than `y_test_predicted_prob_logit`.


In [ ]:
# Exercise 3: Business impact calculation

# Given parameters
contact_cost <- 5             # euros per call
subscription_revenue <- 100   # euros per subscription
total_customers <- 10000
n_test <- length(y_test)
thresholds <- round(seq(0.01, 0.50, by = 0.01), 2)

# Your solution:



### Exercise 4: Odds ratios in business terms

**Task:** Explain three coefficients to a marketing manager.

**Instructions:**
1. From `odds_table`, extract the odds ratios of `poutcome_success`, `contact_telephone` and `cons_conf_idx`.
2. Write one sentence per feature that says what the odds ratio means, stating the "one unit" (having the characteristic, or one standard deviation).
3. Sort the model's features into three groups: (a) the one the marketing manager controls directly, (b) client characteristics she can target on but cannot change, (c) the macro variable she cannot influence, except by timing the campaign.

**Deliverable:** three sentences and the three groups.


In [ ]:
# Exercise 4: Your code here
# Hint: odds_table %>% filter(feature %in% c(...))

# Your solution:



### Exercise 5: Three clients

**Task:** Predict the subscription probability of three clients and decide whom to call.

The clients are described only by features the model has (after section 0.5 the model knows nothing about age or job):

- **Client A:** married, university degree, has a housing loan, no personal loan, reached on a mobile phone, subscribed in the previous campaign, consumer confidence at its average.
- **Client B:** single, high-school education, no housing loan, no personal loan, reached on a landline (`telephone`), never contacted before, consumer confidence one standard deviation below average.
- **Client C:** divorced, basic 9-year education, has a housing loan and a personal loan, reached on a mobile phone, did not subscribe in the previous campaign, consumer confidence at its average.

**Instructions:**
1. Encode each client as one row with the columns of `X_train` (all dummies 0 except the ones that apply). The reference levels `divorced`, `basic.4y`, housing `no`, loan `no`, `cellular` and poutcome `failure` are all zeros. Client A is encoded for you.
2. Compute the deployment probability for each client (log-odds from the model plus `log(sampling_ratio)`, then the sigmoid).
3. Rank the clients and apply the €5/€100 rule from Exercise 3.

**Deliverable:** a table with the three probabilities, and one sentence per client saying whether the bank should call and why.


In [ ]:
# Exercise 5: Three clients, encoded with the columns of X_train
customers <- as.data.frame(matrix(0, nrow = 3, ncol = ncol(X_train),
                                  dimnames = list(c("A", "B", "C"), names(X_train))))

# Client A: married, university degree, housing loan, no personal loan, cellular, previous campaign success, confidence average
customers["A", c("marital_married", "education_university.degree", "housing_yes", "poutcome_success")] <- 1

# Client B and Client C: your encoding here

# Your solution:



### Reflection Questions

After completing the exercises, consider these questions:

1. **When would you prefer LPM over Logistic Regression?**
   - Use the "When the LPM is fine" bullets in section 1 and your result from Exercise 1

2. **How would you explain the difference between these models to a non-technical business stakeholder?**
   - Focus on practical implications rather than mathematical details

3. **What are the key business considerations when choosing a classification threshold?**
   - Think about the costs of false positives (wasted calls) versus false negatives (missed sales), and what Exercise 3 showed when the call cost changed

4. **How might you improve model performance further?**
   - Consider feature engineering (for example recoding `pdays`), different algorithms, or ensemble methods

5. **What ethical considerations should you keep in mind when using these models for customer targeting?**
   - Think about fairness, privacy, and potential discrimination

### Additional Resources

For further learning:
- Practice with different datasets (e.g., credit approval, employee retention)
- Explore other classification algorithms (Random Forest, SVM)
- Learn about feature selection and engineering techniques
- Study advanced evaluation metrics (precision-recall curves, calibration plots)
- Read about cost-sensitive learning and probability calibration as alternatives to undersampling
